In [ ]:
import os
import re
import glob
import random
import warnings
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
from scipy.signal import butter, filtfilt, iirnotch, welch
from scipy.stats import skew, kurtosis

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import f1_score, classification_report
from lightgbm import LGBMClassifier, early_stopping

try:
    from catboost import CatBoostClassifier
    HAS_CATBOOST = True
except ImportError:
    HAS_CATBOOST = False

warnings.filterwarnings("ignore")

# =====================================================================
# 0. CONFIGURATION & REPRODUCIBILITY
# =====================================================================
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Running V6 Physiological Notch+Bandpass & Polar Anatomy Pipeline on: {DEVICE}")

EMG_COLS = [f'ch{i}' for i in range(1, 9)]
IMU_COLS = ['ax', 'ay', 'az', 'gx', 'gy', 'gz']
SIGNAL_COLS = EMG_COLS + IMU_COLS

GESTURES = ['rest', 'fist', 'open', 'point', 'pinch', 'thumb-up']
GESTURE_MAP = {g: i for i, g in enumerate(GESTURES)}
INV_GESTURE_MAP = {i: g for i, g in enumerate(GESTURES)}

FS = 250.0             # Official Aura hardware sampling rate (250 Hz)
WINDOW_SEC = 1.4       # Official test window duration (1.4 s)
SEQ_LEN = 350          # Exact 1.4s * 250 Hz = 350 samples per window
EDGE_TRIM_SEC = 0.20   # Official Aura_Net edge trim to remove human reaction lag
TRAIN_STRIDE_SEC = 0.35
BATCH_SIZE = 64
EPOCHS = 35
LR = 1.0e-3
N_FOLDS = 5

DATA_DIR = None
for root, dirs, files in os.walk('/kaggle/input'):
    if any(f.startswith('train_') and f.endswith('.xlsx') for f in files):
        DATA_DIR = os.path.dirname(root) if os.path.basename(root) == 'train' else root
        break
if DATA_DIR is None:
    DATA_DIR = '/kaggle/input/competitions/vr-calibration-free-emg-challenge-pieeg'

def natural_key(path):
    nums = re.findall(r'\d+', os.path.basename(path))
    return int(nums[0]) if nums else 0

def read_aura_excel_fast(fpath):
    xl = pd.ExcelFile(fpath, engine='openpyxl')
    sheet = 'multimodal' if 'multimodal' in xl.sheet_names else xl.sheet_names[0]
    df = xl.parse(sheet)
    df.columns = [str(c).replace('#', '').strip() for c in df.columns]
    return fpath, df

# =====================================================================
# 1. EXACT 250 HZ RESAMPLING, NOTCH (50/100 HZ), BANDPASS & CHANNEL QC
# =====================================================================
# Zero-phase digital filters at exact FS = 250 Hz
B_NOTCH50, A_NOTCH50 = iirnotch(50.0, Q=30.0, fs=FS)
B_NOTCH100, A_NOTCH100 = iirnotch(100.0, Q=30.0, fs=FS)
B_BP, A_BP = butter(4, [20.0 / (FS / 2.0), 115.0 / (FS / 2.0)], btype='bandpass')
B_IMU_LP, A_IMU_LP = butter(3, 12.0 / (FS / 2.0), btype='lowpass')

# Angles of the 8 equidistant electrodes around the forearm circumference
THETA_8 = np.linspace(0, 2 * np.pi, 8, endpoint=False).astype(np.float32)
COS_8 = np.cos(THETA_8)
SIN_8 = np.sin(THETA_8)

def condition_window_250hz(df_win):
    """
    Processes each 1.4s window 100% independently:
      1. Reconstructs exact sub-millisecond time axis `t` and resamples onto uniform 250 Hz (350 samples).
      2. Mean-centers every channel (strictly matching Test v2).
      3. Applies 50 Hz & 100 Hz Notch filters + 4th-order 20-115 Hz Butterworth Bandpass on EMG.
      4. Performs Per-Window Channel QC (repairs railed/dead electrodes via circular neighbor interpolation).
      5. Smooths IMU dynamic acceleration & angular velocity with a 12 Hz low-pass filter.
    """
    raw_signals = df_win[SIGNAL_COLS].astype(np.float64).values
    raw_signals = pd.DataFrame(raw_signals).interpolate(method='linear', limit_direction='both').fillna(0.0).values
    N_raw = len(raw_signals)

    # Reconstruct strictly increasing time vector t in [0, 1.4]
    if 't' in df_win.columns:
        t_vals = pd.to_numeric(df_win['t'], errors='coerce').ffill().bfill().fillna(0.0).values.astype(np.float64)
        t_vals = t_vals - t_vals[0]
        # Resolve duplicate timestamps from burst packets by micro-stepping monotonic time
        if len(np.unique(t_vals)) < max(5, N_raw // 6) or t_vals[-1] <= 0.05:
            t_vals = np.linspace(0.0, WINDOW_SEC, N_raw)
        else:
            # Spread identical timestamps evenly across the burst interval
            for i in range(1, N_raw):
                if t_vals[i] <= t_vals[i - 1]:
                    t_vals[i] = t_vals[i - 1] + 1e-4
            t_vals = t_vals * (WINDOW_SEC / max(t_vals[-1], 1e-3))
    else:
        t_vals = np.linspace(0.0, WINDOW_SEC, N_raw)

    uniform_t = np.linspace(0.0, WINDOW_SEC, SEQ_LEN)
    resampled = np.zeros((SEQ_LEN, 14), dtype=np.float64)
    for c in range(14):
        resampled[:, c] = np.interp(uniform_t, t_vals, raw_signals[:, c])

    # Strictly mean-center per 1.4s window (matches Test Set v2 guarantee)
    resampled = resampled - np.mean(resampled, axis=0, keepdims=True)

    emg = resampled[:, :8]
    imu = resampled[:, 8:]

    # Apply 50 Hz Notch -> 100 Hz Harmonic Notch -> 20-115 Hz 4th-Order Bandpass
    emg = filtfilt(B_NOTCH50, A_NOTCH50, emg, axis=0)
    emg = filtfilt(B_NOTCH100, A_NOTCH100, emg, axis=0)
    emg = filtfilt(B_BP, A_BP, emg, axis=0)

    # Per-Window Channel Quality Control (Detect railed/saturated or dead electrodes)
    ch_std = np.std(emg, axis=0)
    med_std = np.median(ch_std) + 1e-5
    bad_channels = np.where((ch_std > 12.0 * med_std) | (ch_std < 0.02 * med_std))[0]
    if 0 < len(bad_channels) <= 3:
        emg_qc = emg.copy()
        for b_idx in bad_channels:
            left_c = (b_idx - 1) % 8
            right_c = (b_idx + 1) % 8
            emg_qc[:, b_idx] = 0.5 * (emg[:, left_c] + emg[:, right_c])
        emg = emg_qc

    # Low-pass filter IMU to remove high-frequency tremor/quantization noise
    imu_clean = filtfilt(B_IMU_LP, A_IMU_LP, imu, axis=0)
    imu_clean = imu_clean - np.mean(imu_clean, axis=0, keepdims=True)

    return emg.astype(np.float32), imu_clean.astype(np.float32)  # (350, 8), (350, 6)

# =====================================================================
# 2. PHYSIOLOGICAL & CIRCUMFERENTIAL ANATOMY FEATURE ENGINE (260 FEATS)
# =====================================================================
def extract_v6_features_and_tensor(df_win):
    emg, imu = condition_window_250hz(df_win)
    T = SEQ_LEN  # 350 samples at 250 Hz

    feats = []

    # --- 1. Per-Channel Time-Domain Biopotential Metrics (64 features) ---
    rms = np.sqrt(np.mean(emg ** 2, axis=0)) + 1e-5
    mav = np.mean(np.abs(emg), axis=0) + 1e-5
    emg_diff1 = np.diff(emg, axis=0, prepend=emg[:1, :])
    emg_diff2 = np.diff(emg_diff1, axis=0, prepend=emg_diff1[:1, :])
    wl = np.sum(np.abs(emg_diff1), axis=0) / float(T) + 1e-5
    p95 = np.percentile(np.abs(emg), 95, axis=0) + 1e-5
    iqr = (np.percentile(emg, 75, axis=0) - np.percentile(emg, 25, axis=0)) + 1e-5

    # Zero-Crossing Rate & Slope Sign Changes (with noise threshold)
    thresh = 0.15 * np.median(rms)
    zc = np.sum((emg[:-1] * emg[1:] < 0) & (np.abs(emg[:-1] - emg[1:]) >= thresh), axis=0) / float(T)
    ssc = np.sum((emg_diff1[:-1] * emg_diff1[1:] < 0) & (np.abs(emg_diff1[:-1] - emg_diff1[1:]) >= thresh), axis=0) / float(T)
    # Willison Amplitude (WAMP)
    wamp = np.sum(np.abs(emg_diff1) > thresh, axis=0) / float(T)

    feats.extend(np.log1p(rms))
    feats.extend(np.log1p(mav))
    feats.extend(np.log1p(wl))
    feats.extend(np.log1p(p95))
    feats.extend(np.log1p(iqr))
    feats.extend(zc)
    feats.extend(ssc)
    feats.extend(wamp)

    # --- 2. Hjorth Parameters: Activity, Mobility & Complexity (24 features) ---
    var_x = np.var(emg, axis=0) + 1e-6
    var_d1 = np.var(emg_diff1, axis=0) + 1e-6
    var_d2 = np.var(emg_diff2, axis=0) + 1e-6
    hjorth_mob = np.sqrt(var_d1 / var_x)
    hjorth_comp = np.sqrt(var_d2 / var_d1) / (hjorth_mob + 1e-5)
    feats.extend(np.log1p(var_x))
    feats.extend(hjorth_mob)
    feats.extend(hjorth_comp)

    # --- 3. Forearm Circumferential Activation Topography & Anatomy Compartments (46 features) ---
    tot_rms = np.sum(rms) + 1e-5
    tot_wl = np.sum(wl) + 1e-5
    rms_prop = rms / tot_rms
    wl_prop = wl / tot_wl
    p95_prop = p95 / (np.sum(p95) + 1e-5)
    mav_prop = mav / (np.sum(mav) + 1e-5)

    feats.extend(rms_prop)
    feats.extend(wl_prop)
    feats.extend(p95_prop)
    feats.extend(mav_prop)

    # Anatomical Compartment Grouping (Indices 0..7 correspond to ch1..ch8):
    # Anterior Flexors (ch3, ch4, ch5 -> idx 2, 3, 4): Fist & Pinch
    # Posterior Extensors (ch1, ch7, ch8 -> idx 0, 6, 7): Open & Point
    # Radial Thumb Aspect (ch2, ch3 -> idx 1, 2): Thumb-Up
    flexor_energy = np.sum(rms_prop[[2, 3, 4]])
    extensor_energy = np.sum(rms_prop[[0, 6, 7]])
    radial_thumb_energy = np.sum(rms_prop[[1, 2]])
    ulnar_energy = np.sum(rms_prop[[5, 6]])

    # Polar Spatial Activation Centroid (X_c, Y_c), Angle, and Resultant Concentration R
    cx = float(np.sum(rms_prop * COS_8))
    cy = float(np.sum(rms_prop * SIN_8))
    polar_r = float(np.sqrt(cx ** 2 + cy ** 2))
    polar_angle = float(np.arctan2(cy, cx))
    spatial_entropy = float(-np.sum(rms_prop * np.log(rms_prop + 1e-6)))

    feats.extend([
        float(np.log1p(tot_rms)),
        float(np.log1p(tot_wl)),
        float(np.log1p(np.max(rms))),
        float(np.log1p(np.min(rms))),
        float(np.log1p(np.median(rms))),
        flexor_energy,
        extensor_energy,
        radial_thumb_energy,
        ulnar_energy,
        float(flexor_energy / (extensor_energy + 1e-4)),
        float(radial_thumb_energy / (extensor_energy + 1e-4)),
        float(radial_thumb_energy / (flexor_energy + 1e-4)),
        cx, cy, polar_r, polar_angle, spatial_entropy,
        float(np.max(rms_prop) / (np.mean(rms_prop) + 1e-5))
    ])

    # Rotation-Invariant Circular Autocorrelation & Sorted Profile (Handles Armband Rotation!)
    feats.extend(np.sort(rms_prop))  # 8 sorted proportions (100% invariant to armband rotation!)
    for shift in [1, 2, 3, 4]:
        feats.append(float(np.sum(rms_prop * np.roll(rms_prop, shift))))

    # Opposing & Adjacent Antagonist Contrasts (16 features)
    for i in range(8):
        feats.append(float(rms_prop[i] - rms_prop[(i + 4) % 8]))
        feats.append(float(rms_prop[i] - rms_prop[(i + 1) % 8]))

    # --- 4. Sustain-to-Onset Contractile Ratio & 3-Phase Envelope Dynamics (36 features) ---
    # Onset: [0 : 0.25N], Sustain: [0.25N : 0.75N], Release: [0.75N : N]
    i_q1, i_q3 = int(0.25 * T), int(0.75 * T)
    rms_onset = np.sqrt(np.mean(emg[:i_q1] ** 2, axis=0)) + 1e-4
    rms_sustain = np.sqrt(np.mean(emg[i_q1:i_q3] ** 2, axis=0)) + 1e-4
    rms_release = np.sqrt(np.mean(emg[i_q3:] ** 2, axis=0)) + 1e-4

    sustain_to_onset = rms_sustain / rms_onset
    sustain_to_release = rms_sustain / rms_release
    feats.extend(np.log1p(sustain_to_onset))
    feats.extend(np.log1p(sustain_to_release))
    feats.extend(rms_sustain / (np.sum(rms_sustain) + 1e-5))
    feats.extend(rms_onset / (np.sum(rms_onset) + 1e-5))
    feats.extend([
        float(np.log1p(np.sum(rms_sustain) / np.sum(rms_onset))),
        float(np.log1p(np.sum(rms_sustain))),
        float(np.log1p(np.sum(rms_onset))),
        float(np.log1p(np.sum(rms_release)))
    ])

    # --- 5. Welch Power Spectral Density (PSD) & Physiological Sub-Bands (40 features) ---
    # Sub-Band 1 (20-45 Hz), Core Firing Band (25-65 Hz), Sub-Band 2 (45-75 Hz), Sub-Band 3 (75-115 Hz)
    freqs, psd = welch(emg, fs=FS, nperseg=min(128, T), axis=0)
    psd = psd + 1e-8
    b1_mask = (freqs >= 20.0) & (freqs < 45.0)
    b_core_mask = (freqs >= 25.0) & (freqs <= 65.0)
    b2_mask = (freqs >= 45.0) & (freqs < 75.0)
    b3_mask = (freqs >= 75.0) & (freqs <= 115.0)

    p_b1 = np.mean(psd[b1_mask], axis=0)
    p_core = np.mean(psd[b_core_mask], axis=0)
    p_b2 = np.mean(psd[b2_mask], axis=0)
    p_b3 = np.mean(psd[b3_mask], axis=0)
    p_tot = p_b1 + p_b2 + p_b3 + 1e-6

    # Mean Frequency (MNF) & Median Frequency (MDF) per channel
    mnf = np.sum(freqs[:, None] * psd, axis=0) / (np.sum(psd, axis=0) + 1e-6)
    cum_psd = np.cumsum(psd, axis=0)
    half_psd = 0.5 * cum_psd[-1, :]
    mdf = np.array([freqs[np.searchsorted(cum_psd[:, c], half_psd[c])] for c in range(8)], dtype=np.float32)

    feats.extend(np.log1p(p_core))
    feats.extend(p_b1 / p_tot)
    feats.extend(p_b2 / p_tot)
    feats.extend(p_b3 / p_tot)
    feats.extend(mnf / 125.0)
    feats.extend(mdf / 125.0)

    # --- 6. Riemannian Tangential Spatial Covariance Matrix (44 features) ---
    norm_e = emg / (rms[None, :] + 1e-4)
    cov_mat = (norm_e.T @ norm_e) / float(T) + 1e-3 * np.eye(8)
    eigvals, eigvecs = np.linalg.eigh(cov_mat)
    eigvals = np.clip(eigvals, 1e-4, None)
    log_cov = eigvecs @ np.diag(np.log(eigvals)) @ eigvecs.T
    feats.extend(log_cov[np.triu_indices(8, k=0)])
    feats.extend(np.log(eigvals))

    # --- 7. IMU Kinematic Norms & EMG-to-Motion Coupling (52 features) ---
    acc = imu[:, :3]
    gyro = imu[:, 3:]
    acc_norm = np.sqrt(np.sum(acc ** 2, axis=1))
    gyro_norm = np.sqrt(np.sum(gyro ** 2, axis=1))

    imu_rms = np.sqrt(np.mean(imu ** 2, axis=0)) + 1e-5
    imu_p95 = np.percentile(np.abs(imu), 95, axis=0)
    imu_max = np.max(imu, axis=0)
    imu_min = np.min(imu, axis=0)
    imu_disp = np.mean(imu[T // 2:], axis=0) - np.mean(imu[:T // 2], axis=0)

    # Axis directional proportions (Separates pronation/supination in thumb-up vs flexion in pinch/point)
    acc_dir_prop = imu_rms[:3] / (np.sum(imu_rms[:3]) + 1e-5)
    gyro_dir_prop = imu_rms[3:] / (np.sum(imu_rms[3:]) + 1e-5)

    feats.extend(np.log1p(imu_rms))
    feats.extend(np.log1p(imu_p95))
    feats.extend(imu_max)
    feats.extend(imu_min)
    feats.extend(imu_disp)
    feats.extend(acc_dir_prop)
    feats.extend(gyro_dir_prop)

    # Static Isometric vs. Dynamic Movement Disambiguation Ratios!
    mean_acc_norm = float(np.mean(acc_norm))
    mean_gyro_norm = float(np.mean(gyro_norm))
    max_gyro_norm = float(np.max(gyro_norm))
    emg_to_gyro_ratio = float(np.log1p(tot_rms) / (np.log1p(mean_gyro_norm) + 0.1))
    emg_to_acc_ratio = float(np.log1p(tot_rms) / (np.log1p(mean_acc_norm) + 0.1))

    feats.extend([
        float(np.log1p(mean_acc_norm)), float(np.log1p(np.max(acc_norm))), float(np.log1p(np.std(acc_norm))),
        float(np.log1p(mean_gyro_norm)), float(np.log1p(max_gyro_norm)), float(np.log1p(np.std(gyro_norm))),
        emg_to_gyro_ratio, emg_to_acc_ratio
    ])

    imu_corr = np.nan_to_num(np.corrcoef(imu.T), nan=0.0)
    feats.extend(imu_corr[np.triu_indices(6, k=1)])

    tab_vec = np.nan_to_num(np.array(feats, dtype=np.float32), nan=0.0, posinf=0.0, neginf=0.0)

    # --- 8. Construct Clean 30-Channel Sequence Tensor (Downsampled 350 -> 175 steps = 125 Hz) ---
    env_win = 15
    emg_env = np.sqrt(pd.DataFrame(emg ** 2).rolling(window=env_win, min_periods=1, center=True).mean().values + 1e-5)
    emg_spatial_t = emg_env / (np.sum(emg_env, axis=1, keepdims=True) + 1e-4)
    emg_log_t = np.sign(emg) * np.log1p(np.abs(emg))
    env_log_t = np.log1p(emg_env)
    imu_log_t = np.sign(imu) * np.log1p(np.abs(imu) * 10.0)

    # 8 (emg_log) + 8 (env_log) + 8 (spatial_ratio) + 6 (imu_log) = 30 channels
    seq_full = np.concatenate([emg_log_t, env_log_t, emg_spatial_t, imu_log_t], axis=1)  # (350, 30)
    # 2x average pooling along time: 350 -> 175 steps
    seq_175 = 0.5 * (seq_full[0::2, :] + seq_full[1::2, :])
    seq_tensor = seq_175.T.astype(np.float32)  # (30, 175)

    return seq_tensor, tab_vec

# =====================================================================
# 3. EDGE-TRIMMED TRAINING WINDOW SLICER (REMOVES 0.2s TRANSITION NOISE)
# =====================================================================
def process_train_session(args):
    session_idx, fpath = args
    _, df = read_aura_excel_fast(fpath)

    if 'label_source' in df.columns:
        df = df[df['label_source'].astype(str).str.strip().str.lower() == 'guided'].reset_index(drop=True)

    label_col = 'label' if 'label' in df.columns else 'gesture'
    df[label_col] = df[label_col].astype(str).str.strip().str.lower()
    df = df[df[label_col].isin(GESTURE_MAP.keys())].reset_index(drop=True)
    if len(df) == 0:
        return None

    time_jump = df['t'].diff().abs() > 0.25
    label_change = df[label_col] != df[label_col].shift()
    df['seg_id'] = (time_jump | label_change).cumsum()

    seq_list, tab_list, cls_list, anchor_list = [], [], [], []

    for _, seg in df.groupby('seg_id'):
        t_start, t_end = float(seg['t'].iloc[0]), float(seg['t'].iloc[-1])
        dur = t_end - t_start
        gesture_id = GESTURE_MAP[seg[label_col].iloc[0]]

        # CRITICAL FIX: Trim 0.20s reaction/release transitions from segment edges (Aura_Net spec)
        if dur >= WINDOW_SEC + 2 * EDGE_TRIM_SEC:
            eff_start = t_start + EDGE_TRIM_SEC
            eff_end = t_end - EDGE_TRIM_SEC
        elif dur >= WINDOW_SEC + 0.15:
            trim = 0.5 * (dur - WINDOW_SEC)
            eff_start = t_start + trim
            eff_end = t_end - trim
        else:
            eff_start = t_start
            eff_end = t_end

        curr_t = eff_start
        w_idx = 0
        while curr_t + WINDOW_SEC <= eff_end + 0.06:
            win = seg[(seg['t'] >= curr_t) & (seg['t'] < curr_t + WINDOW_SEC)]
            if len(win) >= 35:
                s_t, t_v = extract_v6_features_and_tensor(win)
                seq_list.append(s_t)
                tab_list.append(t_v)
                cls_list.append(gesture_id)
                anchor_list.append(1 if (w_idx % 3 == 0) else 0)
                w_idx += 1
            curr_t += TRAIN_STRIDE_SEC

    if not seq_list:
        return None

    n = len(seq_list)
    return (
        np.stack(seq_list),
        np.stack(tab_list),
        np.array(cls_list, dtype=np.int64),
        np.full(n, session_idx, dtype=np.int64),
        np.array(anchor_list, dtype=np.int64)
    )

def load_v6_all_data(data_dir):
    train_files = sorted(glob.glob(os.path.join(data_dir, "**", "train_*.xlsx"), recursive=True), key=natural_key)
    test_files = sorted(glob.glob(os.path.join(data_dir, "**", "test_*.xlsx"), recursive=True), key=natural_key)
    print(f"Processing {len(train_files)} train sessions (with 0.2s edge trim & 50/100Hz notch) and {len(test_files)} test files...")

    with ThreadPoolExecutor(max_workers=4) as ex:
        train_res = list(ex.map(process_train_session, enumerate(train_files)))
        test_loaded = list(ex.map(read_aura_excel_fast, test_files))

    train_res = [r for r in train_res if r is not None]
    X_seq = np.concatenate([r[0] for r in train_res], axis=0)
    X_tab = np.concatenate([r[1] for r in train_res], axis=0)
    y_cls = np.concatenate([r[2] for r in train_res], axis=0)
    groups = np.concatenate([r[3] for r in train_res], axis=0)
    is_anchor = np.concatenate([r[4] for r in train_res], axis=0)

    X_test_seq, X_test_tab, test_ids = [], [], []
    for fpath, df in test_loaded:
        id_col = 'action_id' if 'action_id' in df.columns else df.columns[0]
        for act_id, win in df.groupby(id_col, sort=False):
            s_t, t_v = extract_v6_features_and_tensor(win)
            X_test_seq.append(s_t)
            X_test_tab.append(t_v)
            test_ids.append(act_id)

    X_test_seq = np.stack(X_test_seq)
    X_test_tab = np.stack(X_test_tab)

    # Global Standardization of Sequence Tensor Channels (Preserves amplitude & spatial ratios!)
    ch_mu = np.mean(X_seq, axis=(0, 2), keepdims=True)
    ch_sd = np.std(X_seq, axis=(0, 2), keepdims=True) + 1e-5
    X_seq = np.clip((X_seq - ch_mu) / ch_sd, -8.0, 8.0)
    X_test_seq = np.clip((X_test_seq - ch_mu) / ch_sd, -8.0, 8.0)

    print(f"Loaded {len(X_seq)} Train Windows (Anchors: {np.sum(is_anchor)}) | Tabular Feats: {X_tab.shape[1]} | Seq: {X_seq.shape}")
    print(f"Loaded {len(X_test_seq)} Independent Test Windows")
    return X_seq, X_tab, y_cls, groups, is_anchor, X_test_seq, X_test_tab, test_ids

# =====================================================================
# 4. AURANET TEMPORAL-SPATIAL 1D-RESNET + BIGRU ARCHITECTURE
# =====================================================================
class AuraV6Dataset(Dataset):
    def __init__(self, X_seq, X_tab, y_cls=None, is_train=True):
        self.X_seq = X_seq
        self.X_tab = X_tab
        self.y_cls = y_cls
        self.is_train = is_train

    def __len__(self):
        return len(self.X_seq)

    def __getitem__(self, idx):
        x = self.X_seq[idx].copy()
        tab = self.X_tab[idx].copy()

        if self.is_train:
            if random.random() < 0.50:
                shift = random.randint(-8, 8)
                x = np.roll(x, shift=shift, axis=-1)
            if random.random() < 0.40:
                scale = np.random.uniform(0.90, 1.10, size=(x.shape[0], 1)).astype(np.float32)
                noise = np.random.normal(0.0, 0.02, size=x.shape).astype(np.float32)
                x = x * scale + noise
            if random.random() < 0.25:
                t0 = random.randint(0, x.shape[1] - 15)
                x[:, t0 : t0 + 15] = 0.0

        if self.y_cls is None:
            return torch.tensor(x, dtype=torch.float32), torch.tensor(tab, dtype=torch.float32)

        return (
            torch.tensor(x, dtype=torch.float32),
            torch.tensor(tab, dtype=torch.float32),
            torch.tensor(self.y_cls[idx], dtype=torch.long)
        )

class ResSEBlock1D(nn.Module):
    def __init__(self, in_c, out_c, k=5, stride=1):
        super().__init__()
        self.conv1 = nn.Conv1d(in_c, out_c, kernel_size=k, stride=stride, padding=k // 2)
        self.bn1 = nn.BatchNorm1d(out_c)
        self.conv2 = nn.Conv1d(out_c, out_c, kernel_size=3, stride=1, padding=1)
        self.bn2 = nn.BatchNorm1d(out_c)
        self.se = nn.Sequential(
            nn.Linear(out_c, out_c // 4),
            nn.GELU(),
            nn.Linear(out_c // 4, out_c),
            nn.Sigmoid()
        )
        self.short = nn.Conv1d(in_c, out_c, 1, stride=stride) if (in_c != out_c or stride != 1) else nn.Identity()

    def forward(self, x):
        h = F.gelu(self.bn1(self.conv1(x)))
        h = self.bn2(self.conv2(h))
        w = self.se(h.mean(dim=-1)).unsqueeze(-1)
        return F.gelu(h * w + self.short(x))

class AuraNetV6(nn.Module):
    def __init__(self, in_channels=30, tab_dim=260, num_classes=6):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(in_channels, 64, kernel_size=7, padding=3),
            nn.BatchNorm1d(64),
            nn.GELU()
        )
        self.res1 = ResSEBlock1D(64, 96, k=5, stride=2)
        self.res2 = ResSEBlock1D(96, 128, k=5, stride=2)
        self.bigru = nn.GRU(128, 96, num_layers=2, batch_first=True, bidirectional=True, dropout=0.2)
        self.attn = nn.Sequential(
            nn.Linear(192, 64),
            nn.Tanh(),
            nn.Linear(64, 1)
        )

        self.tab_mlp = nn.Sequential(
            nn.LayerNorm(tab_dim),
            nn.Dropout(0.25),
            nn.Linear(tab_dim, 192),
            nn.BatchNorm1d(192),
            nn.GELU(),
            nn.Dropout(0.20),
            nn.Linear(192, 128),
            nn.BatchNorm1d(128),
            nn.GELU()
        )

        # 192 (Attn) + 192 (Max) + 128 (Tab) = 512
        self.head = nn.Sequential(
            nn.LayerNorm(512),
            nn.Dropout(0.30),
            nn.Linear(512, 128),
            nn.GELU(),
            nn.Dropout(0.15),
            nn.Linear(128, num_classes)
        )

    def forward(self, x_seq, x_tab):
        h = self.stem(x_seq)
        h = self.res2(self.res1(h))
        seq, _ = self.bigru(h.permute(0, 2, 1))

        w = F.softmax(self.attn(seq), dim=1)
        attn_p = torch.sum(seq * w, dim=1)
        max_p = torch.max(seq, dim=1)[0]
        tab_e = self.tab_mlp(x_tab)

        return self.head(torch.cat([attn_p, max_p, tab_e], dim=-1))

class FocalMacroF1Loss(nn.Module):
    def __init__(self, num_classes=6, smoothing=0.05, gamma=1.5, f1_w=0.35):
        super().__init__()
        self.num_classes = num_classes
        self.smoothing = smoothing
        self.gamma = gamma
        self.f1_w = f1_w

    def forward(self, logits, targets):
        probs = F.softmax(logits, dim=-1)
        log_probs = F.log_softmax(logits, dim=-1)
        with torch.no_grad():
            true_dist = torch.full_like(log_probs, self.smoothing / (self.num_classes - 1))
            true_dist.scatter_(1, targets.unsqueeze(1), 1.0 - self.smoothing)

        focal = torch.mean(torch.sum(-true_dist * ((1.0 - probs) ** self.gamma) * log_probs, dim=-1))

        one_hot = F.one_hot(targets, num_classes=self.num_classes).float()
        tp = torch.sum(probs * one_hot, dim=0)
        fp = torch.sum(probs * (1.0 - one_hot), dim=0)
        fn = torch.sum((1.0 - probs) * one_hot, dim=0)
        soft_f1 = (2.0 * tp + 1e-5) / (2.0 * tp + fp + fn + 1e-5)

        return (1.0 - self.f1_w) * focal + self.f1_w * (1.0 - torch.mean(soft_f1))

def predict_auranet_tta(model, loader, device):
    model.eval()
    out = []
    with torch.no_grad():
        for batch in loader:
            bx_s, bx_t = batch[0].to(device), batch[1].to(device)
            l0 = model(bx_s, bx_t)
            l1 = model(torch.roll(bx_s, shifts=-4, dims=-1), bx_t)
            l2 = model(torch.roll(bx_s, shifts=4, dims=-1), bx_t)
            p = 0.50 * F.softmax(l0, dim=-1) + 0.25 * F.softmax(l1, dim=-1) + 0.25 * F.softmax(l2, dim=-1)
            out.append(p.cpu().numpy())
    return np.concatenate(out, axis=0)

# =====================================================================
# 5. 5-FOLD TRAINING: LIGHTGBM + CATBOOST + EXTRATREES + AURANET-V6
# =====================================================================
X_seq, X_tab, y_cls, groups, is_anchor, X_test_seq, X_test_tab, test_ids = load_v6_all_data(DATA_DIR)

sgkf = StratifiedGroupKFold(n_splits=N_FOLDS)

oof_lgb = np.zeros((len(X_seq), 6), dtype=np.float32)
oof_cat = np.zeros((len(X_seq), 6), dtype=np.float32)
oof_et  = np.zeros((len(X_seq), 6), dtype=np.float32)
oof_nn  = np.zeros((len(X_seq), 6), dtype=np.float32)

test_lgb = np.zeros((len(X_test_seq), 6), dtype=np.float32)
test_cat = np.zeros((len(X_test_seq), 6), dtype=np.float32)
test_et  = np.zeros((len(X_test_seq), 6), dtype=np.float32)
test_nn  = np.zeros((len(X_test_seq), 6), dtype=np.float32)

criterion = FocalMacroF1Loss()

for fold, (trn_idx, val_idx) in enumerate(sgkf.split(X_seq, y_cls, groups)):
    print(f"\n==================== FOLD {fold + 1} / {N_FOLDS} ====================")
    clean_val_idx = val_idx[is_anchor[val_idx] == 1]
    anchor_submask = (is_anchor[val_idx] == 1)

    scaler = RobustScaler()
    X_tab_trn = np.clip(scaler.fit_transform(X_tab[trn_idx]), -10.0, 10.0)
    X_tab_val = np.clip(scaler.transform(X_tab[val_idx]), -10.0, 10.0)
    X_tab_val_clean = np.clip(scaler.transform(X_tab[clean_val_idx]), -10.0, 10.0)
    X_tab_tst = np.clip(scaler.transform(X_test_tab), -10.0, 10.0)

    # --- 1. LightGBM on 250Hz Notch+Bandpass Physiological Features ---
    lgb_model = LGBMClassifier(
        n_estimators=1500,
        learning_rate=0.025,
        num_leaves=31,
        max_depth=6,
        min_child_samples=25,
        subsample=0.80,
        subsample_freq=1,
        colsample_bytree=0.65,
        reg_alpha=1.5,
        reg_lambda=3.0,
        random_state=42 + fold,
        n_jobs=-1,
        verbose=-1
    )
    lgb_model.fit(
        X_tab_trn, y_cls[trn_idx],
        eval_set=[(X_tab_val_clean, y_cls[clean_val_idx])],
        callbacks=[early_stopping(100, verbose=False)]
    )
    oof_lgb[val_idx] = lgb_model.predict_proba(X_tab_val)
    test_lgb += lgb_model.predict_proba(X_tab_tst) / N_FOLDS
    f1_l = f1_score(y_cls[clean_val_idx], np.argmax(oof_lgb[clean_val_idx], axis=1), average='macro')

    # --- 2. CatBoost Classifier ---
    if HAS_CATBOOST:
        cat_model = CatBoostClassifier(
            iterations=1600,
            learning_rate=0.035,
            depth=6,
            l2_leaf_reg=5.0,
            random_seed=42 + fold,
            task_type='CPU',
            verbose=False
        )
        cat_model.fit(
            X_tab_trn, y_cls[trn_idx],
            eval_set=(X_tab_val_clean, y_cls[clean_val_idx]),
            early_stopping_rounds=100,
            verbose=False
        )
        oof_cat[val_idx] = cat_model.predict_proba(X_tab_val)
        test_cat += cat_model.predict_proba(X_tab_tst) / N_FOLDS
        f1_c = f1_score(y_cls[clean_val_idx], np.argmax(oof_cat[clean_val_idx], axis=1), average='macro')
    else:
        oof_cat[val_idx] = oof_lgb[val_idx]
        test_cat += test_lgb / N_FOLDS
        f1_c = f1_l

    # --- 3. ExtraTrees Classifier (High-Variance Reduction on Riemannian & Spectral Features) ---
    et_model = ExtraTreesClassifier(
        n_estimators=500,
        max_depth=16,
        min_samples_leaf=4,
        max_features='sqrt',
        class_weight='balanced',
        random_state=42 + fold,
        n_jobs=-1
    )
    et_model.fit(X_tab_trn, y_cls[trn_idx])
    oof_et[val_idx] = et_model.predict_proba(X_tab_val)
    test_et += et_model.predict_proba(X_tab_tst) / N_FOLDS
    f1_e = f1_score(y_cls[clean_val_idx], np.argmax(oof_et[clean_val_idx], axis=1), average='macro')

    # --- 4. AuraNetV6 Deep Waveform + Tabular Hybrid ---
    trn_ds = AuraV6Dataset(X_seq[trn_idx], X_tab_trn, y_cls[trn_idx], is_train=True)
    val_ds = AuraV6Dataset(X_seq[val_idx], X_tab_val, is_train=False)
    tst_ds = AuraV6Dataset(X_test_seq, X_tab_tst, is_train=False)

    trn_loader = DataLoader(trn_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
    tst_loader = DataLoader(tst_ds, batch_size=BATCH_SIZE, shuffle=False)

    model = AuraNetV6(in_channels=X_seq.shape[1], tab_dim=X_tab.shape[1]).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1.5e-2)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)

    best_nn_f1 = 0.0
    best_nn_p = None
    ckpt = f"auranet_v6_fold{fold}.pt"

    for epoch in range(1, EPOCHS + 1):
        model.train()
        for bx_s, bx_t, by_c in trn_loader:
            bx_s, bx_t, by_c = bx_s.to(DEVICE), bx_t.to(DEVICE), by_c.to(DEVICE)
            optimizer.zero_grad()

            if random.random() < 0.30:
                lam = np.random.beta(0.4, 0.4)
                perm = torch.randperm(bx_s.size(0)).to(DEVICE)
                logits = model(lam * bx_s + (1 - lam) * bx_s[perm], lam * bx_t + (1 - lam) * bx_t[perm])
                loss = lam * criterion(logits, by_c) + (1 - lam) * criterion(logits, by_c[perm])
            else:
                logits = model(bx_s, bx_t)
                loss = criterion(logits, by_c)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        scheduler.step()

        vp = predict_auranet_tta(model, val_loader, DEVICE)
        cf1 = f1_score(y_cls[val_idx][anchor_submask], np.argmax(vp[anchor_submask], axis=1), average='macro')
        if cf1 > best_nn_f1:
            best_nn_f1 = cf1
            best_nn_p = vp
            torch.save(model.state_dict(), ckpt)

    oof_nn[val_idx] = best_nn_p
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    test_nn += predict_auranet_tta(model, tst_loader, DEVICE) / N_FOLDS

    print(f"  Fold {fold+1} Clean Val Macro-F1 -> LGBM: {f1_l:.4f} | CatBoost: {f1_c:.4f} | ExtraTrees: {f1_e:.4f} | AuraNetV6: {best_nn_f1:.4f}")

# =====================================================================
# 6. OPTIMAL ENSEMBLE BLENDING & THRESHOLD TUNING
# =====================================================================
clean_mask = (is_anchor == 1)
y_clean = y_cls[clean_mask]

print("\n========================================================")
print(f"Clean OOF Macro-F1 (LGBM Alone):       {f1_score(y_clean, np.argmax(oof_lgb[clean_mask], axis=1), average='macro'):.4f}")
print(f"Clean OOF Macro-F1 (CatBoost Alone):   {f1_score(y_clean, np.argmax(oof_cat[clean_mask], axis=1), average='macro'):.4f}")
print(f"Clean OOF Macro-F1 (ExtraTrees Alone): {f1_score(y_clean, np.argmax(oof_et[clean_mask], axis=1), average='macro'):.4f}")
print(f"Clean OOF Macro-F1 (AuraNetV6 Alone):  {f1_score(y_clean, np.argmax(oof_nn[clean_mask], axis=1), average='macro'):.4f}")

best_ens_f1 = 0.0
best_w = (0.30, 0.30, 0.15, 0.25)

for w_l in np.linspace(0.10, 0.50, 9):
    for w_c in np.linspace(0.10, 0.50, 9):
        for w_e in np.linspace(0.0, 0.30, 7):
            w_n = max(0.0, 1.0 - w_l - w_c - w_e)
            if w_l + w_c + w_e > 1.0:
                continue
            blend = (
                w_l * oof_lgb[clean_mask] +
                w_c * oof_cat[clean_mask] +
                w_e * oof_et[clean_mask] +
                w_n * oof_nn[clean_mask]
            )
            s = f1_score(y_clean, np.argmax(blend, axis=1), average='macro')
            if s > best_ens_f1:
                best_ens_f1 = s
                best_w = (w_l, w_c, w_e, w_n)

oof_blend = best_w[0] * oof_lgb + best_w[1] * oof_cat + best_w[2] * oof_et + best_w[3] * oof_nn
test_blend = best_w[0] * test_lgb + best_w[1] * test_cat + best_w[2] * test_et + best_w[3] * test_nn

class_mult = np.ones(6, dtype=np.float32)
tuned_f1 = best_ens_f1
for _ in range(3):
    for c in range(6):
        for m in np.linspace(0.85, 1.20, 36):
            cand = class_mult.copy()
            cand[c] = m
            s = f1_score(y_clean, np.argmax(oof_blend[clean_mask] * cand[None, :], axis=1), average='macro')
            if s > tuned_f1:
                tuned_f1 = s
                class_mult = cand

print(f"Clean OOF Macro-F1 (4-Model Ensemble): {best_ens_f1:.4f} | Weights (LGB, CAT, ET, NN): {np.round(best_w, 2)}")
print(f"Clean OOF Macro-F1 (Tuned Ensemble):   {tuned_f1:.4f} | Multipliers: {np.round(class_mult, 3)}")
print("========================================================")
print(classification_report(y_clean, np.argmax(oof_blend[clean_mask] * class_mult[None, :], axis=1), target_names=GESTURES))

final_preds = np.argmax(test_blend * class_mult[None, :], axis=1)
final_labels = [INV_GESTURE_MAP[i] for i in final_preds]

sample_sub_files = glob.glob(os.path.join(DATA_DIR, "**", "sample_submission*.csv"), recursive=True)
if sample_sub_files:
    sub = pd.read_csv(sample_sub_files[0])
    id_col, target_col = sub.columns[0], sub.columns[1]
    if len(sub) == len(final_labels) and len(set(test_ids)) == len(test_ids):
        pred_map = dict(zip(test_ids, final_labels))
        sub[target_col] = sub[id_col].map(pred_map).fillna(pd.Series(final_labels))
    else:
        sub[target_col] = final_labels
else:
    sub = pd.DataFrame({'action_id': test_ids, 'gesture': final_labels})

sub.to_csv("submission.csv", index=False)
sub.to_csv("/kaggle/working/submission.csv", index=False)
print("\nSaved V6 submission.csv! Test Set Prediction Distribution:")
print(sub.iloc[:, 1].value_counts())